# Cohort Mental Health Comparison: ABCD vs GUSTO

Verify that the ABCD and GUSTO cohorts have comparable mental health distributions
before pooling or replication. Tests for distributional similarity (Mann-Whitney U)
and variance homogeneity (Fligner-Killeen) on internalizing and externalizing scores.

## Configuration

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# All paths are relative to the repository root.
# Set BASE_DIR to the repo root if running from a different working directory.

ABCD_MH_CSV  = 'data/outcomes/mhabcd.csv'    # cols: mh_p_cbcl__synd__int_sum, mh_p_cbcl__synd__ext_sum
GUSTO_MH_CSV = 'data/external/mhgusto.csv'   # cols: ysr_int, ysr_ext
OUTPUT_DIR   = 'outputs/cohort_comparison'

## Setup: Imports and Output Directory

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, fligner

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory ready: {OUTPUT_DIR}")

## Step 1: Load Data

In [ ]:
abcd  = pd.read_csv(ABCD_MH_CSV)
gusto = pd.read_csv(GUSTO_MH_CSV)

print(f"ABCD  shape : {abcd.shape}")
print(f"GUSTO shape : {gusto.shape}")
print()
print("ABCD columns  :", abcd.columns.tolist())
print("GUSTO columns :", gusto.columns.tolist())

## Step 2: Z-Score Within Each Cohort

In [ ]:
# Z-score internalizing and externalizing within each cohort separately
abcd['int_z']  = (abcd['mh_p_cbcl__synd__int_sum'] - abcd['mh_p_cbcl__synd__int_sum'].mean()) / abcd['mh_p_cbcl__synd__int_sum'].std()
abcd['ext_z']  = (abcd['mh_p_cbcl__synd__ext_sum'] - abcd['mh_p_cbcl__synd__ext_sum'].mean()) / abcd['mh_p_cbcl__synd__ext_sum'].std()

gusto['int_z'] = (gusto['ysr_int'] - gusto['ysr_int'].mean()) / gusto['ysr_int'].std()
gusto['ext_z'] = (gusto['ysr_ext'] - gusto['ysr_ext'].mean()) / gusto['ysr_ext'].std()

print("ABCD  int_z  — mean: {:.4f}, std: {:.4f}".format(abcd['int_z'].mean(),  abcd['int_z'].std()))
print("GUSTO int_z  — mean: {:.4f}, std: {:.4f}".format(gusto['int_z'].mean(), gusto['int_z'].std()))
print("ABCD  ext_z  — mean: {:.4f}, std: {:.4f}".format(abcd['ext_z'].mean(),  abcd['ext_z'].std()))
print("GUSTO ext_z  — mean: {:.4f}, std: {:.4f}".format(gusto['ext_z'].mean(), gusto['ext_z'].std()))

## Step 3: Mann-Whitney U Test (Distribution Comparison)

In [ ]:
# Mann-Whitney U: ABCD vs GUSTO on z-scored internalizing and externalizing
stat_int, p_int = mannwhitneyu(abcd['int_z'].dropna(), gusto['int_z'].dropna(), alternative='two-sided')
stat_ext, p_ext = mannwhitneyu(abcd['ext_z'].dropna(), gusto['ext_z'].dropna(), alternative='two-sided')

print("Mann-Whitney U — Internalizing (z-scored)")
print(f"  U = {stat_int:.2f},  p = {p_int:.6f}")
print()
print("Mann-Whitney U — Externalizing (z-scored)")
print(f"  U = {stat_ext:.2f},  p = {p_ext:.6f}")

## Step 4: Fligner-Killeen Variance Test (Raw Values)

In [ ]:
# Fligner-Killeen test on raw (non-z-scored) values to assess variance homogeneity
fl_stat_int, fl_p_int = fligner(
    abcd['mh_p_cbcl__synd__int_sum'].dropna(),
    gusto['ysr_int'].dropna()
)
fl_stat_ext, fl_p_ext = fligner(
    abcd['mh_p_cbcl__synd__ext_sum'].dropna(),
    gusto['ysr_ext'].dropna()
)

print("Fligner-Killeen — Internalizing (raw)")
print(f"  stat = {fl_stat_int:.4f},  p = {fl_p_int:.6e}")
print()
print("Fligner-Killeen — Externalizing (raw)")
print(f"  stat = {fl_stat_ext:.4f},  p = {fl_p_ext:.6e}")

## Step 5: Save Summary Table

In [ ]:
summary = pd.DataFrame([
    {
        'outcome':       'Internalizing',
        'abcd_n':        int(abcd['mh_p_cbcl__synd__int_sum'].notna().sum()),
        'gusto_n':       int(gusto['ysr_int'].notna().sum()),
        'abcd_mean_raw': abcd['mh_p_cbcl__synd__int_sum'].mean(),
        'gusto_mean_raw':gusto['ysr_int'].mean(),
        'abcd_std_raw':  abcd['mh_p_cbcl__synd__int_sum'].std(),
        'gusto_std_raw': gusto['ysr_int'].std(),
        'mwu_stat':      stat_int,
        'mwu_p':         p_int,
        'fligner_stat':  fl_stat_int,
        'fligner_p':     fl_p_int,
    },
    {
        'outcome':       'Externalizing',
        'abcd_n':        int(abcd['mh_p_cbcl__synd__ext_sum'].notna().sum()),
        'gusto_n':       int(gusto['ysr_ext'].notna().sum()),
        'abcd_mean_raw': abcd['mh_p_cbcl__synd__ext_sum'].mean(),
        'gusto_mean_raw':gusto['ysr_ext'].mean(),
        'abcd_std_raw':  abcd['mh_p_cbcl__synd__ext_sum'].std(),
        'gusto_std_raw': gusto['ysr_ext'].std(),
        'mwu_stat':      stat_ext,
        'mwu_p':         p_ext,
        'fligner_stat':  fl_stat_ext,
        'fligner_p':     fl_p_ext,
    },
])

out_path = os.path.join(OUTPUT_DIR, 'mh_cohort_comparison.csv')
summary.to_csv(out_path, index=False)
print(f"Summary saved to {out_path}")
print()
print(summary.to_string(index=False))